In [ ]:
import numpy as np
import time
import os
import matplotlib.pyplot as plt
from numba import njit, prange

# ==========================================
# 1. SYSTEM PARAMETERS & SWEEP TARGETS
# ==========================================
N_ATOMS = 5000  
BOX_SIZE = 1.0
K_SPRING = 5000.0
MASS = 1.0

DT_INIT = 0.001
DT_MAX = 0.01
N_MIN = 5
F_INC = 1.1
F_DEC = 0.5
ALPHA_START = 0.1
F_ALPHA = 0.99
TOL = 1e-4

MAX_STEPS = 100000        
CAPTURE_STEP = 150  
MAX_WALL_TIME = 20.0  # Cutoff at 20 seconds to keep the 25-run sweep moving

# Define the 5 Densities
PHI_SWEEP = [0.80, 0.82, 0.84, 0.86, 0.90]

# Define the 5 Target Boundary Fractions (Gamma)
GAMMA_TARGETS = [0.05, 0.10, 0.15, 0.20, 0.25]

# ==========================================
# 2. NUMBA JIT-COMPILED CORE PHYSICS 
# ==========================================
@njit(parallel=True)
def get_forces_mpt_numba(pos, atom_dt, radius, k_spring):
    N = len(pos)
    forces = np.zeros_like(pos)
    forces_dt = np.zeros_like(pos)
    cutoff = 2.0 * radius
    for i in prange(N):
        for j in range(N):
            if i == j: continue
            
            dx_raw = pos[i, 0] - pos[j, 0]
            dy_raw = pos[i, 1] - pos[j, 1]
            
            dx = dx_raw - np.round(dx_raw)
            dy = dy_raw - np.round(dy_raw)
            
            dist = np.sqrt(dx**2 + dy**2)
            if dist < cutoff:
                f_mag = k_spring * (cutoff - dist)
                if dist > 1e-12:
                    fx = (dx / dist) * f_mag
                    fy = (dy / dist) * f_mag
                else:
                    fx = 0.0
                    fy = 0.0
                
                forces[i, 0] += fx
                forces[i, 1] += fy
                dt_ij = min(atom_dt[i], atom_dt[j])
                forces_dt[i, 0] += fx * dt_ij
                forces_dt[i, 1] += fy * dt_ij
    return forces, forces_dt

@njit(parallel=True)
def get_total_energy_numba(pos, radius, k_spring):
    N = len(pos)
    total_energy = 0.0
    cutoff = 2.0 * radius
    for i in prange(N):
        for j in range(i + 1, N):
            dx_raw = pos[i, 0] - pos[j, 0]
            dy_raw = pos[i, 1] - pos[j, 1]
            dx = dx_raw - np.round(dx_raw)
            dy = dy_raw - np.round(dy_raw)
            dist = np.sqrt(dx**2 + dy**2)
            if dist < cutoff:
                overlap = cutoff - dist
                total_energy += 0.5 * k_spring * overlap**2
    return total_energy

def map_atoms_to_grid(pos, divs):
    ix = np.clip(np.floor(pos[:, 0] * divs).astype(int), 0, divs-1)
    iy = np.clip(np.floor(pos[:, 1] * divs).astype(int), 0, divs-1)
    return ix * divs + iy

# ==========================================
# 3. TIMED FIRE ENGINES
# ==========================================
def run_global_fire(pos_init, max_steps, radius):
    pos = np.copy(pos_init)
    vel = np.zeros_like(pos)
    dt = DT_INIT
    alpha = ALPHA_START
    npos = 0
    
    energy_history = []
    dt_history = []
    time_history = []
    
    t0 = time.time()
    for step in range(max_steps):
        if step % 10 == 0:
            elapsed = time.time() - t0
            if elapsed > MAX_WALL_TIME:
                break
                
            e_curr = get_total_energy_numba(pos, radius, K_SPRING)
            energy_history.append(e_curr)
            dt_history.append(dt)
            time_history.append(elapsed)
            
        atom_dt = np.full(N_ATOMS, dt)
        forces, forces_dt = get_forces_mpt_numba(pos, atom_dt, radius, K_SPRING)
        
        vel += 0.5 * forces_dt / MASS
        pos += vel * dt
        pos = np.mod(pos, BOX_SIZE)
        
        forces_new, forces_dt_new = get_forces_mpt_numba(pos, atom_dt, radius, K_SPRING)
        vel += 0.5 * forces_dt_new / MASS
        forces = forces_new
        
        P_global = np.sum(forces * vel)
        v_mag = np.linalg.norm(vel, axis=1, keepdims=True)
        f_mag = np.linalg.norm(forces, axis=1, keepdims=True)
        mask = (f_mag > 1e-12).flatten()
        
        if np.any(mask):
            vel[mask] = (1 - alpha) * vel[mask] + alpha * (forces[mask]/f_mag[mask]) * v_mag[mask]
            
        if P_global > 0:
            npos += 1
            if npos > N_MIN:
                dt = min(dt * F_INC, DT_MAX)
                alpha *= F_ALPHA
        else:
            npos = 0
            dt *= F_DEC
            vel[:, :] = 0.0
            alpha = ALPHA_START
            
        if np.sqrt(np.sum(forces**2) / (2 * N_ATOMS)) < TOL:
            break
            
    return pos, np.array(energy_history), np.array(dt_history), np.array(time_history), time.time() - t0

def run_async_fire(pos_init, max_steps, capture_step, radius, grid_divs):
    pos = np.copy(pos_init)
    vel = np.zeros_like(pos)
    n_domains = grid_divs * grid_divs
    
    d_dt = np.full(n_domains, DT_INIT)
    d_alpha = np.full(n_domains, ALPHA_START)
    d_npos = np.zeros(n_domains, dtype=int)
    
    energy_history = []
    dt_mean_history = []
    time_history = []
    snapshot_dt = None
    accumulated_dt = np.zeros(n_domains)
    
    atom_indices = map_atoms_to_grid(pos, grid_divs)
    forces, forces_dt = get_forces_mpt_numba(pos, d_dt[atom_indices], radius, K_SPRING)
    
    t0 = time.time()
    for step in range(max_steps):
        accumulated_dt += d_dt
        
        if step % 10 == 0:
            elapsed = time.time() - t0
            if elapsed > MAX_WALL_TIME:
                if snapshot_dt is None: snapshot_dt = np.copy(d_dt)
                break
                
            e_current = get_total_energy_numba(pos, radius, K_SPRING)
            energy_history.append(e_current)
            dt_mean_history.append(np.mean(d_dt))
            time_history.append(elapsed)
            
        if step == capture_step:
            snapshot_dt = np.copy(d_dt)
            
        vel += 0.5 * forces_dt / MASS
        pos += vel * d_dt[atom_indices][:, np.newaxis]
        pos = np.mod(pos, BOX_SIZE)
        
        atom_indices = map_atoms_to_grid(pos, grid_divs)
        forces_new, forces_dt_new = get_forces_mpt_numba(pos, d_dt[atom_indices], radius, K_SPRING)
        vel += 0.5 * forces_dt_new / MASS
        forces = forces_new; forces_dt = forces_dt_new
        
        p_domain = np.bincount(atom_indices, weights=np.sum(forces * vel, axis=1), minlength=n_domains)
        
        atom_alpha = d_alpha[atom_indices][:, np.newaxis]
        v_mag = np.linalg.norm(vel, axis=1, keepdims=True)
        f_mag = np.linalg.norm(forces, axis=1, keepdims=True)
        mask = (f_mag > 1e-12).flatten()
        if np.any(mask): 
            vel[mask] = (1 - atom_alpha[mask]) * vel[mask] + atom_alpha[mask] * (forces[mask]/f_mag[mask]) * v_mag[mask]
            
        mask_up = p_domain > 0
        d_npos[mask_up] += 1
        mask_grow = mask_up & (d_npos > N_MIN)
        d_dt[mask_grow] = np.minimum(d_dt[mask_grow] * F_INC, DT_MAX)
        d_alpha[mask_grow] *= F_ALPHA
        
        mask_down = p_domain <= 0
        d_npos[mask_down] = 0; d_dt[mask_down] *= F_DEC; d_alpha[mask_down] = ALPHA_START
        
        vel[mask_down[atom_indices]] = 0.0
        
        if np.sqrt(np.sum(forces**2) / (2 * N_ATOMS)) < TOL: 
            if snapshot_dt is None: snapshot_dt = np.copy(d_dt)
            break
            
    if snapshot_dt is None: snapshot_dt = np.copy(d_dt)
    
    return pos, snapshot_dt, np.array(energy_history), np.array(dt_mean_history), np.array(time_history), time.time() - t0, accumulated_dt


# ==========================================
# 4. THE 5x5 RENDER CELL
# ==========================================
def render_sweep_results():
    print("\n" + "="*70)
    print("Generating the 5x5 Parameter Sweep Dashboard...")
    
    fig, axes = plt.subplots(len(PHI_SWEEP), len(GAMMA_TARGETS), figsize=(24, 20), sharex='col', sharey='row')
    fig.suptitle(f"Async FIRE vs Global FIRE: Energy Convergence vs. Wall-Clock Time (N={N_ATOMS})", fontsize=22, y=0.95)
    
    for i, phi in enumerate(PHI_SWEEP):
        current_radius = np.sqrt((phi * BOX_SIZE**2) / (N_ATOMS * np.pi))
        k_values = [max(1, int(np.round(g * BOX_SIZE / (8 * current_radius)))) for g in GAMMA_TARGETS]
        k_values = sorted(list(set(k_values)))
        
        # Ensure we pad k_values if rounding collapsed some targets
        while len(k_values) < len(GAMMA_TARGETS):
            k_values.append(k_values[-1] + 2)
            
        for j, grid in enumerate(k_values[:len(GAMMA_TARGETS)]):
            ax = axes[i, j]
            filename = f"parameter_sweep/sweep_phi{phi:.2f}_grid{grid}.npz"
            
            if os.path.exists(filename):
                data = np.load(filename)
                t_glob = data['t_hist_glob']
                e_glob = data['e_hist_glob']
                t_async = data['t_hist_async']
                e_async = data['e_hist_async']
                gamma_val = data['gamma'] if 'gamma' in data else (8 * current_radius * grid) / BOX_SIZE
                
                ax.plot(t_glob, e_glob, 'k--', linewidth=2, label='Global Baseline')
                ax.plot(t_async, e_async, 'r-', linewidth=2, label='Async FIRE')
                
                ax.set_yscale('log')
                ax.set_title(f"$\phi$={phi:.2f} | K={grid}x{grid} ($\gamma$={gamma_val:.2f})", fontsize=14)
                ax.grid(alpha=0.3)
                
                if i == len(PHI_SWEEP) - 1:
                    ax.set_xlabel("Wall-Clock Time (s)", fontsize=12)
                if j == 0:
                    ax.set_ylabel("Total Potential Energy", fontsize=12)
                if i == 0 and j == 0:
                    ax.legend(loc='upper right')
            else:
                ax.text(0.5, 0.5, "Data Not Generated Yet", ha='center', va='center', transform=ax.transAxes, color='gray')
                ax.set_title(f"$\phi$={phi:.2f} | K={grid}x{grid}", fontsize=14)
                ax.set_xticks([])
                ax.set_yticks([])

    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    out_img = "parameter_sweep_results_5x5.pdf"
    plt.savefig(out_img, bbox_inches='tight')
    print(f"Success! Dashboard rendered and saved as '{out_img}'.")
    plt.show()


# ==========================================
# 5. AUTOMATED PARAMETER SPACE RUNNER
# ==========================================
if __name__ == '__main__':
    os.makedirs("parameter_sweep", exist_ok=True)
    np.random.seed(42)
    base_positions = np.random.rand(N_ATOMS, 2)

    print("Warming up Numba JIT compiler...")
    dummy_r = np.sqrt((0.82 * BOX_SIZE**2) / (10 * np.pi))
    _ = get_forces_mpt_numba(base_positions[:10], np.ones(10), dummy_r, K_SPRING)
    _ = get_total_energy_numba(base_positions[:10], dummy_r, K_SPRING)

    print(f"\nInitiating automated 25-run parameter landscape evaluation loop...")
    print("Press Ctrl+C at any time to safely stop the sweep and instantly render the partial data.")
    
    try:
        for phi_idx, phi in enumerate(PHI_SWEEP):
            current_radius = np.sqrt((phi * BOX_SIZE**2) / (N_ATOMS * np.pi))
            
            k_values = [max(1, int(np.round(g * BOX_SIZE / (8 * current_radius)))) for g in GAMMA_TARGETS]
            k_values = sorted(list(set(k_values)))
            
            while len(k_values) < len(GAMMA_TARGETS):
                k_values.append(k_values[-1] + 2)
            k_values = k_values[:len(GAMMA_TARGETS)]
            
            print(f"\n[DENSITY PHASE {phi_idx+1}/5] Evaluating phi = {phi:.2f} (Radius: {current_radius:.5f})")
            print(" Running Global FIRE baseline for this configuration...")
            pos_glob, e_hist_glob, dt_hist_glob, t_hist_glob, time_glob = run_global_fire(base_positions, MAX_STEPS, current_radius)
            
            for grid_idx, grid in enumerate(k_values):
                actual_gamma = (8 * current_radius * grid) / BOX_SIZE
                print(f"   -> Running Async FIRE on {grid}x{grid} grid (Gamma: {actual_gamma:.3f})...")
                
                pos_async, dt_grid_snap, e_hist_async, dt_hist_async, t_hist_async, time_async, accumulated_dt = run_async_fire(
                    base_positions, MAX_STEPS, CAPTURE_STEP, current_radius, grid
                )
                
                out_filename = f"parameter_sweep/sweep_phi{phi:.2f}_grid{grid}.npz"
                np.savez_compressed(out_filename, 
                                    pos_async=pos_async, pos_glob=pos_glob, dt_grid_snap=dt_grid_snap, 
                                    e_hist_async=e_hist_async, e_hist_glob=e_hist_glob,
                                    dt_hist_async=dt_hist_async, dt_hist_glob=dt_hist_glob,
                                    t_hist_async=t_hist_async, t_hist_glob=t_hist_glob,
                                    time_async=time_async, time_glob=time_glob,
                                    accumulated_dt=accumulated_dt, radius=current_radius,
                                    k_spring=K_SPRING, grid_divs=grid, n_atoms=N_ATOMS,
                                    box_size=BOX_SIZE, phi=phi, gamma=actual_gamma)
                
                print(f"      [SAVED] File: '{out_filename}'")
                print(f"      Global Final Energy: {e_hist_glob[-1]:.4e} | Async Final Energy: {e_hist_async[-1]:.4e}")

        print("\nOptimization parameter sweep finished successfully!")
        
    except KeyboardInterrupt:
        print("\n" + "!"*70)
        print("KEYBOARD INTERRUPT DETECTED. Halting simulation engines early.")
        print("Safely preserving existing files and transitioning to Render Phase...")
        print("!"*70)

    # Automatically call the rendering function at the end (whether completed or interrupted)
    render_sweep_results()